**K-Nearest Neighbors (k-NN)** K-Nearest Neighbors algorithm is a non-parametric, instance-based supervised learning method used for classification and regression. It skips an explicit training phase, instead storing the entire training dataset in a multi-dimensional vector space. When predicting a novel data point, the algorithm computes the geometric distance—commonly using Euclidean metrics—to locate the k closest historical instances. The final classification is determined through a majority vote of these neighboring instances, while regression tasks compute their average value.

In [1]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the Iris plant dataset
iris = load_iris()
X = iris.data  # Features: sepal length/width, petal length/width
y = iris.target  # Labels: 0 (Setosa), 1 (Versicolor), 2 (Virginica)

# 2. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Scale features (Crucial for k-NN distance calculations!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and train the k-NN Classifier
# We use 3 neighbors; weights='uniform' means all neighbors vote equally
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_scaled, y_train)

# 5. Make predictions on the test set
y_pred = knn.predict(X_test_scaled)

# 6. Evaluate the results
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed Performance Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Overall Accuracy: 93.33%

Detailed Performance Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.83      1.00      0.91        10
   virginica       1.00      0.80      0.89        10

    accuracy                           0.93        30
   macro avg       0.94      0.93      0.93        30
weighted avg       0.94      0.93      0.93        30



In [5]:
print("Features", X_test[0], "output ", y_test[0]) #as output is zero so it is setosa plant
print("predicted class", knn.predict([X_test_scaled[0]]))

Features [4.4 3.  1.3 0.2] output  0
predicted class [0]


**Support Vector Machine (SVM)** Support Vector Machine is a powerful supervised learning model designed to analyze data for classification, regression, and outlier detection. The algorithm operates by mapping data points into a high-dimensional feature space to identify an optimal separating decision boundary known as a hyperplane. It explicitly maximizes the geometric margin, which is the maximum distance between the hyperplane and the closest training samples, termed support vectors. To resolve complex, non-linear relationships, SVM applies the kernel trick to implicitly project the data into higher dimensions without incurring prohibitive computational costs.

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Load the Iris plant dataset
iris = load_iris()
X = iris.data
y = iris.target

# 2. Split into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Scale features (Crucial for SVM to maximize boundaries accurately)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Initialize and train the Support Vector Classifier
# 'kernel="rbf"' allows SVM to draw curved, non-linear boundaries
svm_model = SVC(kernel='rbf', C=1.0, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# 5. Make predictions on the test set
y_pred = svm_model.predict(X_test_scaled)

# 6. Evaluate the results
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Detailed SVM Performance Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Overall Accuracy: 96.67%

Detailed SVM Performance Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [7]:
print("Features", X_test[0], "output ", y_test[0]) #as output is zero so it is setosa plant
print("predicted class", knn.predict([X_test_scaled[0]]))

Features [4.4 3.  1.3 0.2] output  0
predicted class [0]


## Task 1: k-NN Boundary Distortion and High-Dimensional Degradation
Part A: The Curse of Fractional Distance Metrics
You are deploying a 3-NN classifier on a genome dataset with 5,000 continuous features, but the target classes are completely differentiated by only 3 specific features. Explain mathematically why standard Euclidean distance (L₂ norm) fails as the feature dimension D → ∞ (the distance convergence phenomenon). Propose how switching to a fractional distance metric ($L_p$ norm where 0 < p < 1) affects the relative contrast between the nearest and furthest neighbors compared to the standard Euclidean metric.
Part B: Decision Boundary Inversion via Weighting Functions
Consider a binary classification problem ($Y \in \{0, 1\}$) in a 2D space. A query point $X_q$ is surrounded by 5 nearest neighbors: the 2 closest neighbors belong to class 0 (at distance d=1), and the next 3 neighbors belong to class 1 (at distance d=2). Contrast the final predicted class probability $P(Y=1\vert{}X_q)$ when using standard uniform voting versus an inverse-distance weighting scheme where weight $w_i = 1 / d_i^2$.
Part C: Computational Complexity and KD-Tree Failures
To optimize query lookup time from $O(N \cdot D)$ to $O(D \cdot \log N)$, you construct a KD-Tree index for your training data. Detail the structural scenario where a KD-Tree's query performance degrades back to a brute-force linear search. Explain how the relationship between the number of training samples N and the number of dimensions D dictates this algorithmic bottleneck.
Part D: Boundary Smoothing via Multi-Scale Evaluation
Describe a programmatic approach using cross-validation to detect if your selected k value is causing the model to overfit to local noise (forming localized "islands" of a class within the decision boundary). How does the ratio of the training set size to the optimal k value change when moving from a low-noise dataset to a highly overlapping, high-variance dataset?
------------------------------
## Task 2: SVM Kernel Dualism and Regularization Mechanics
Part A: Non-Mercer Kernels and Optimization Failure
A developer attempts to create a custom SVM similarity metric defined as $K(x, z) = \sin(x^T z)$. Explain from a convex optimization perspective why this function fails to satisfy Mercer’s Condition. Describe what happens to the objective function of the Dual SVM optimization problem (Lagrangian formulation) and the underlying Gram matrix when a non-positive semi-definite kernel is forced into the solver.
Part B: Asymmetric Cost Margins for Class Imbalance
You are training a Support Vector Classifier (SVC) with an RBF kernel on a highly imbalanced dataset where the positive class represents only 1% of the data. Instead of utilizing a single regularization hyperparameter C, you implement an asymmetric soft-margin formulation using C₊ for positive misclassifications and C₋ for negative misclassifications. Mathematically or conceptually explain how adjusting the ratio C₊ / C₋ shifts the separating hyperplane and changes the geometric composition of the support vectors.
Part C: Dual vs. Primal Computational Inversion
You must train a Linear SVM on two different setups: Setup 1 has 500,000 samples and 20 features ($N \gg D$). Setup 2 has 2,000 samples and 80,000 features ($D \gg N$). Explain which formulation—Primal or Dual—should be selected for each setup to optimize training time and memory footprint, referencing how the dimensions of the optimization variables depend on N and D.
Part D: Geometric Interpretation of RBF Gamma Extremes
Analyze the geometric behavior of the decision boundary of an RBF Kernel SVM as the hyperparameter γ (gamma) approaches infinity (γ → ∞), and as it approaches zero (γ → 0). Specifically, describe what happens to the radius of influence of individual support vectors and how this structurally impacts the variance and bias of the system.


